<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 02 · REAL-TIME ANALYTICS WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Make Doris Scan Less Data</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">Use the same 10,158,080 event rows to compare sort keys, partition pruning, bucket (tablet) pruning, Query Profile scan evidence, and repeated-key behavior.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Doris 4.1.3 · Sort Key · Prefix Index · Partition · Tablet · Query Profile</span>
</div>

This lab continues from Lab 1. It creates two derived internal tables from `doris_course.events`; it does not read from or write to S3. Every large-table comparison uses the same rows, so a smaller scan scope reflects physical design rather than a smaller dataset. Run the cells in order.

### Initialize the Lab

Run the next cell before Section 1. It reloads the shared helper, installs the same output styles used in Lab 1, and creates the `lab` object used by later cells. Run it again after every Jupyter kernel restart. It does not start Docker or change any table.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT);

## 1. Establish the physical-design baseline

The notebook reconnects to the existing single-node integrated Doris sandbox and selects `doris_course`. The first result verifies the row count and time span. `SHOW CREATE TABLE` then reveals what Doris selected when Lab 1 omitted an explicit key and distribution clause.

In this course environment, the baseline resolves to a table using the Duplicate Key model, with sort key `(event_time, event_id, user_id)` and `RANDOM BUCKETS AUTO`. The actual bucket count is inspected in Section 2.

The three key columns were inferred; they are not a primary key. When no table model or key clause is given, Doris normally selects the Duplicate Key model and derives key columns from eligible columns at the beginning of the schema. These key columns form the sort key, while their leading prefix forms the Prefix Index. The automatic selection stops at three columns, 36 index bytes, an unsupported Prefix Index type, or after including a `VARCHAR` column. For this schema, the first three fixed-length columns become `(event_time, event_id, user_id)`.

In the Duplicate Key model, these columns determine storage order and the Prefix Index, but repeated key values are retained. This differs from a MySQL `PRIMARY KEY`, which identifies a row and rejects duplicate key values. Doris uses the Unique Key model when one logical row per key and upsert behavior are required.

Reference: [Duplicate Key model](https://doris.apache.org/docs/4.x/table-design/data-model/duplicate/) · [CREATE TABLE](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/table-and-view/table/CREATE-TABLE/)

In [ ]:
lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")

lab.sql("""
SELECT
    COUNT(*) AS row_count,
    MIN(event_time) AS min_event_time,
    MAX(event_time) AS max_event_time
FROM events
""", title="Baseline data check")

lab.sql(
    "SHOW CREATE TABLE events",
    title="Baseline physical design",
    metadata_view="design",
);

**Expected result**

`Baseline data check` returns one row:

| row_count | min_event_time | max_event_time |
|---:|---|---|
| 10,158,080 | 2019-12-01 00:00:12 | 2020-03-11 04:53:08 |

`Baseline physical design` returns the same six design choices shown by the compact result table:

| design choice | resolved value |
|---|---|
| Table model | Duplicate Key model |
| Sort key | event_time, event_id, user_id |
| Partitioning | No explicit partitioning |
| Distribution | RANDOM |
| Buckets per partition | AUTO |
| Replica allocation | 1 replica per tablet |

The physical number selected by `AUTO` is inspected in Section 2. If the baseline table is missing or its row count differs, complete Lab 1 before continuing.

## 2. Map the table to its physical storage hierarchy

Doris names data logically through Catalog, Database, and Table. Inside an internal table, a Partition contains one Tablet for each Bucket. A load transaction can create a Rowset in each affected Tablet, and each Rowset contains immutable columnar Segment files. Background Compaction merges small Rowsets to limit read amplification.

```text
internal Catalog
└── doris_course Database
    └── events Table
        └── Partition              logical range and lifecycle boundary
            └── Bucket / Tablet    distribution rule / physical data shard
                └── Rowset         version unit created by a load transaction or Compaction
                    └── Segment    immutable columnar data file
```

`SHOW PARTITIONS` exposes Partition and Bucket metadata. `SHOW TABLETS` exposes Tablets and their replicas. `VersionCount` indicates visible data versions; it is useful Compaction evidence, but it is not a direct count of Rowsets or Segments.

The DDL in Section 1 records the bucket-count policy as `AUTO`. The `Buckets` column returned below records the resulting physical layout. In the current course sandbox, Doris resolves that policy to 10 buckets for `events`. This is an observed result for this table and environment, not a universal default.

In [ ]:
lab.sql(
    "SHOW PARTITIONS FROM events",
    title="Baseline partitions",
    columns=["PartitionName", "State", "Buckets", "DistributionKey", "RowCount"],
)

lab.sql(
    "SHOW TABLETS FROM events ORDER BY LocalDataSize DESC LIMIT 10",
    title="Baseline tablets",
    columns=[
        "TabletId", "BackendId", "LocalDataSize",
        "RowCount", "VersionCount", "VisibleVersionCount",
    ],
);

**Expected result**

- `events` has one unpartitioned/default partition and 10 tablets in this sandbox.
- Every tablet has one replica because the All-in-One environment has one BE and uses one replica.
- The tablets contain different row subsets, but Random bucketing provides no bucket key that a `user_id` equality predicate can prune.

## 3. Isolate the effect of the leading sort key

`events_v2` keeps one Partition and 10 Random Buckets, but moves `user_id` to the leading sort-key position. This isolates the effect of the sort key and Prefix Index. The key columns in the Duplicate Key model define storage order here; they do not enforce uniqueness.

The table is truncated before the insert so every run rebuilds the same 10,158,080 rows from the Lab 1 internal table. This keeps the comparison repeatable and exposes the time required to write the new physical layout.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS events_v2 (
    user_id BIGINT NOT NULL,
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12, 2) NOT NULL DEFAULT "0.00"
)
DUPLICATE KEY(user_id, event_time, event_id)
DISTRIBUTED BY RANDOM BUCKETS 10
PROPERTIES (
    "replication_num" = "1"
)
""")

lab.execute("TRUNCATE TABLE events_v2")

lab.insert("""
INSERT INTO events_v2 (
    user_id, event_time, event_id, event_type, region, product_id, revenue
)
SELECT
    user_id, event_time, event_id, event_type, region, product_id, revenue
FROM events
""", title="Build the user-first sort layout")

lab.sql("""
SELECT 'events' AS table_name, COUNT(*) AS row_count, SUM(revenue) AS total_revenue
FROM events
UNION ALL
SELECT 'events_v2', COUNT(*), SUM(revenue)
FROM events_v2
ORDER BY table_name
""", title="v1 and v2 content check");

**Expected result**

The insert result first reports 10,158,080 affected rows and the time required to build `events_v2`. Both validation rows then show `row_count = 10,158,080` and `total_revenue = 39,984,455.64`. The table contents agree; only the column order and sort key differ.

## 4. Measure how the leading sort key reduces scan work

The four queries form two controlled pairs. Within each pair, the predicate and aggregate are identical and only the table changes. Query Profile records runtime evidence from the BE scan operator. The notebook temporarily disables SQL Cache, Query Cache, and Condition Cache so that every statement performs a new scan, then restores the original session settings.

Inside each tablet, Doris stores rows in sort-key order and builds a sparse Prefix Index over the leading key prefix. The index narrows the row ranges that the scan must examine; it does not enforce uniqueness and does not by itself remove partitions or tablets.

- `events` is ordered by `(event_time, event_id, user_id)`, so the time range matches its leading key. A `user_id` filter cannot use that leading prefix when `event_time` and `event_id` are unconstrained.
- `events_v2` is ordered by `(user_id, event_time, event_id)`, so the equality filter on `user_id` matches its leading key. A time-only filter no longer matches the leading prefix.

Read the Profile columns as follows:

| Metric | Meaning in this comparison |
|---|---|
| `event_count`, `total_revenue` | Query result; matching values establish semantic equivalence. |
| `rows_read` | Rows returned by the scan after scan-side filtering. It should remain the same within each query pair. |
| `scan_rows` | Rows examined from storage. This is the primary row-volume evidence for the sort-key comparison. |
| `scan_bytes` | Bytes examined by the scan. It supports `scan_rows`, but encoding, selected columns, and Compaction can affect the exact value. |

The exact scan volume can vary after Compaction and across machines. Compare the direction and scale of the paired values instead of expecting an identical byte count.

In [ ]:
lab.compare_profiles([
    ("events · user filter", """
        SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events
        WHERE user_id = 610871788
    """),
    ("events_v2 · user filter", """
        SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events_v2
        WHERE user_id = 610871788
    """),
    ("events · time filter", """
        SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events
        WHERE event_time >= '2020-03-01 00:00:00'
          AND event_time <  '2020-03-02 00:00:00'
    """),
    ("events_v2 · time filter", """
        SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events_v2
        WHERE event_time >= '2020-03-01 00:00:00'
          AND event_time <  '2020-03-02 00:00:00'
    """),
], title="Sort-key runtime comparison");

**Expected result**

| Query | Expected `event_count` | Expected `total_revenue` | Expected `rows_read` | Expected scan observation |
|---|---:|---:|---:|---|
| `events · user filter` | 200 | 24,192.36 | 200 | `scan_rows` remains high because `user_id` is not the leading key. |
| `events_v2 · user filter` | 200 | 24,192.36 | 200 | `scan_rows` is substantially lower because `user_id` is the leading key. |
| `events · time filter` | 75,259 | 5,670,241.29 | 75,259 | `scan_rows` is low because `event_time` is the leading key. |
| `events_v2 · time filter` | 75,259 | 5,670,241.29 | 75,259 | `scan_rows` is substantially higher because `event_time` no longer leads the key. |

The result and `rows_read` of each pair should match. The difference appears in `scan_rows` and `scan_bytes`: placing one access pattern first in the sort key can make a different access pattern less efficient.

## 5. Enable partition pruning and bucket (tablet) pruning

`events_v3` keeps the baseline sort key, adds daily Auto Partitioning on `event_time`, and distributes every partition by `HASH(user_id)`. Doris creates a partition when inserted rows first require that day. The source contains events on 26 distinct days, so the completed table should contain 26 partitions and 260 tablets: 10 tablets in each partition.

The table is truncated before insertion so Auto Partitioning and the full physical layout are rebuilt from the unchanged baseline on every run. The displayed insert duration therefore measures a real write rather than a rerun guard.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS events_v3 (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12, 2) NOT NULL DEFAULT "0.00"
)
DUPLICATE KEY(event_time, event_id, user_id)
AUTO PARTITION BY RANGE (date_trunc(event_time, 'day'))
()
DISTRIBUTED BY HASH(user_id) BUCKETS 10
PROPERTIES (
    "replication_num" = "1"
)
""")

lab.execute("TRUNCATE TABLE events_v3")

lab.insert("""
INSERT INTO events_v3 (
    event_time, event_id, user_id, event_type, region, product_id, revenue
)
SELECT
    event_time, event_id, user_id, event_type, region, product_id, revenue
FROM events
""", title="Build the partitioned and hash-bucketed layout")

lab.sql("""
SELECT 'events' AS table_name, COUNT(*) AS row_count, SUM(revenue) AS total_revenue
FROM events
UNION ALL
SELECT 'events_v3', COUNT(*), SUM(revenue)
FROM events_v3
ORDER BY table_name
""", title="v1 and v3 content check");

**Expected result**

The insert result first reports 10,158,080 affected rows and the time required to build `events_v3`. Both tables then contain 10,158,080 rows and total revenue of 39,984,455.64. Auto Partitioning changes placement, not the logical dataset.

## 6. Connect load transactions to Tablet storage and Compaction

Use three Doris metadata statements to connect the table definition to its physical layout. Each statement remains visible in the code cell, and its result presents a compact view of the relevant metadata.

- The design summary extracts the table model, sort key, partition rule, distribution rule, bucket count, and replica allocation.
- The partition summary verifies what Auto Partitioning actually created from the event dates.
- The tablet summary covers every tablet and reports ranges and totals instead of selecting an arbitrary ten-row sample.

In the tablet summary, `Unique tablets` counts physical data shards, `Replicas per tablet` confirms the sandbox replication factor, `Rows per tablet` shows the distribution range, and `Local replica size` sums the storage occupied by local replicas.

A load transaction publishes immutable Rowsets inside the affected tablets, and each Rowset contains one or more columnar Segments. Background Compaction later merges Rowsets within a tablet without changing query results. `version count` is useful operational evidence, but it is not a direct count of Rowsets or Segments and is not a fixed expected value.

In [ ]:
lab.sql(
    "SHOW CREATE TABLE events_v3",
    title="v3 physical design",
    metadata_view="design",
)

lab.sql(
    "SHOW PARTITIONS FROM events_v3 ORDER BY PartitionName",
    title="v3 partition layout",
    metadata_view="partitions",
)

lab.sql(
    "SHOW TABLETS FROM events_v3",
    title="v3 tablet and version overview",
    metadata_view="tablets",
);

**Expected result**

- The design summary reports the Duplicate Key model, sort key `(event_time, event_id, user_id)`, daily Auto Range partitioning, `HASH(user_id)`, 10 Buckets per Partition, and one replica.
- The partition summary reports 26 healthy daily partitions, 260 total tablets, and 10,158,080 rows. Its bounds run from 2019-12-01 through the exclusive upper bound 2020-03-12; Doris creates partitions only for days that occur in the source data.
- The tablet summary reports 260 unique tablets and one replica per tablet. It summarizes all tablets rather than displaying an arbitrary subset.
- Local size, rows per tablet, and `VersionCount` can vary with distribution and background Compaction. Interpret their ranges; do not grade exact values.

## 7. Prove that FE pruning reduces the scan scope

Predicate pushdown and pruning answer different questions. A pushed predicate is evaluated by the BE scan operator close to storage. Partition pruning and bucket (tablet) pruning happen earlier: FE uses table metadata to exclude physical objects from the distributed query plan.

The next cell compares the same three predicate shapes on `events` and `events_v3`. Its result table keeps only the selected/total ratios for `partitions` and `tablets`; generated partition names and the complete plans remain available in a collapsed output.

In [ ]:
lab.compare_explain([
    ("events · time", """
        EXPLAIN SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events
        WHERE event_time >= '2020-03-01 00:00:00'
          AND event_time <  '2020-03-02 00:00:00'
    """),
    ("events_v3 · time", """
        EXPLAIN SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events_v3
        WHERE event_time >= '2020-03-01 00:00:00'
          AND event_time <  '2020-03-02 00:00:00'
    """),
    ("events · user", """
        EXPLAIN SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events
        WHERE user_id = 610871788
    """),
    ("events_v3 · user", """
        EXPLAIN SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events_v3
        WHERE user_id = 610871788
    """),
    ("events · time + user", """
        EXPLAIN SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events
        WHERE event_time >= '2020-03-01 00:00:00'
          AND event_time <  '2020-03-02 00:00:00'
          AND user_id = 610871788
    """),
    ("events_v3 · time + user", """
        EXPLAIN SELECT COUNT(*) AS event_count, SUM(revenue) AS total_revenue
        FROM events_v3
        WHERE event_time >= '2020-03-01 00:00:00'
          AND event_time <  '2020-03-02 00:00:00'
          AND user_id = 610871788
    """),
], title="Partition pruning and bucket (tablet) pruning");

**Expected result**

| Predicate | `events` baseline | `events_v3` |
|---|---|---|
| Time only | `partitions=1/1`, `tablets=10/10` | approximately `partitions=1/26`, with all 10 tablets in the selected day |
| User only | `partitions=1/1`, `tablets=10/10` | all 26 partitions, but approximately one Hash tablet per partition |
| Time + user | `partitions=1/1`, `tablets=10/10` | approximately one partition and one tablet |

The denominator may be presented as the table-wide total, such as 260, rather than the per-partition total. In either form, a numerator smaller than its denominator demonstrates pruning. All six plans should still show the relevant scan predicate, which demonstrates predicate pushdown independently of pruning.

## 8. Verify that pruning does not change query results

Section 7 used `EXPLAIN` to compare the planned Partition and Bucket/Tablet scan scope. `EXPLAIN` does not execute a query or return its analytical result.

This section therefore runs the combined-filter `SELECT` against both tables. The two result rows must be identical: `events_v3` changes where Doris reads the data, not the data or the meaning of the query.

In [ ]:
lab.sql("""
SELECT 'events' AS table_name, COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM events
WHERE event_time >= '2020-03-01 00:00:00'
  AND event_time <  '2020-03-02 00:00:00'
  AND user_id = 610871788
UNION ALL
SELECT 'events_v3', COUNT(*), SUM(revenue)
FROM events_v3
WHERE event_time >= '2020-03-01 00:00:00'
  AND event_time <  '2020-03-02 00:00:00'
  AND user_id = 610871788
ORDER BY table_name
""", title="Combined-filter result check");

**Expected result**

Both rows report 11 events and total revenue of 4,592.14. `events_v3` reaches the same result with a smaller planned scan scope.

## 9. Explain repeated-key behavior with table models

These small tables receive the same two write transactions. The first writes one row for users 1001 and 1002. The second writes another row for the existing key `(1001, 2026-01-01)`. Keeping the input identical makes the model behavior visible without another large import.

- The Duplicate Key model preserves all detail rows.
- The Unique Key model exposes the latest row for the key.
- The Aggregate Key model combines value columns according to their declared aggregation function; this example uses `SUM`.

For `model_agg`, `revenue DECIMAL(12, 2) SUM` declares how rows with the same aggregate key values are combined. Doris therefore exposes one logical `revenue` value for each `(user_id, event_date)` when it is selected directly. The final comparison runs an explicit `SUM(revenue) GROUP BY user_id, event_date` over those logical rows and places both query forms side by side.

In [ ]:
for table_name in ("model_dup", "model_uniq", "model_agg"):
    lab.execute(f"DROP TABLE IF EXISTS {table_name}")

lab.execute("""
CREATE TABLE model_dup (
    user_id BIGINT NOT NULL,
    event_date DATE NOT NULL,
    revenue DECIMAL(12, 2) NOT NULL
)
DUPLICATE KEY(user_id, event_date)
DISTRIBUTED BY HASH(user_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")

lab.execute("""
CREATE TABLE model_uniq (
    user_id BIGINT NOT NULL,
    event_date DATE NOT NULL,
    revenue DECIMAL(12, 2) NOT NULL
)
UNIQUE KEY(user_id, event_date)
DISTRIBUTED BY HASH(user_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")

lab.execute("""
CREATE TABLE model_agg (
    user_id BIGINT NOT NULL,
    event_date DATE NOT NULL,
    revenue DECIMAL(12, 2) SUM NOT NULL
)
AGGREGATE KEY(user_id, event_date)
DISTRIBUTED BY HASH(user_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")

for table_name in ("model_dup", "model_uniq", "model_agg"):
    lab.execute(f"""
        INSERT INTO {table_name} VALUES
            (1001, '2026-01-01', 10.00),
            (1002, '2026-01-01', 20.00)
    """)
    lab.execute(f"INSERT INTO {table_name} VALUES (1001, '2026-01-01', 15.00)")

lab.sql("""
SELECT 'duplicate' AS model, user_id, event_date, revenue
FROM model_dup
UNION ALL
SELECT 'unique', user_id, event_date, revenue
FROM model_uniq
UNION ALL
SELECT 'aggregate', user_id, event_date, revenue
FROM model_agg
ORDER BY model, user_id, revenue
""", title="Repeated-key results")

lab.sql("""
SELECT
    'Direct model value' AS query_form,
    user_id,
    event_date,
    revenue AS revenue_result
FROM model_agg

UNION ALL

SELECT
    'Explicit SUM with GROUP BY' AS query_form,
    user_id,
    event_date,
    SUM(revenue) AS revenue_result
FROM model_agg
GROUP BY user_id, event_date

ORDER BY user_id, query_form
""", title="Aggregate Key model query equivalence");

**Expected result**

`Repeated-key results` shows how the same writes produce different logical results:

| Model | Result for user 1001 | Total visible rows |
|---|---|---:|
| Duplicate Key model | two rows: 10.00 and 15.00 | 3 |
| Unique Key model | latest value: 15.00 | 2 |
| Aggregate Key model | summed value: 25.00 | 2 |

`Aggregate Key model query equivalence` shows both query forms for each aggregate key:

| query_form | user_id | event_date | revenue_result |
|---|---:|---|---:|
| Direct model value | 1001 | 2026-01-01 | 25.00 |
| Explicit SUM with GROUP BY | 1001 | 2026-01-01 | 25.00 |
| Direct model value | 1002 | 2026-01-01 | 20.00 |
| Explicit SUM with GROUP BY | 1002 | 2026-01-01 | 20.00 |

The values match because the query groups by the complete aggregate key and applies the same `SUM` function declared for `revenue`. Doris already combines repeated-key values when producing the Aggregate Key model's logical rows; the explicit query aggregates those logical rows again without changing this result.

This equivalence is specific to this grouping and aggregation. A query that groups by a different key set or calculates another metric still needs its own aggregation. At production scale, the Aggregate Key model is most useful when many source rows share the same keys, because those rows can be represented by fewer aggregated logical rows.

### Stop the Doris sandbox

Run this optional cell when you have finished working and want to release the container's CPU and memory. It stops the Doris processes but keeps the container, image, FE metadata volume, and BE storage volume. Skip it if you want to continue using the sandbox.

In [ ]:
lab.shell(r"""
set -euo pipefail

docker stop doris
docker inspect --format 'container={{.State.Status}}' doris
""", title="Stop the Doris sandbox");

**Expected result:** Docker reports `container=exited`. The baseline and derived tables remain in the named volumes.

### Restart the Doris sandbox

Run this cell when you want to continue with the same Lab 2 tables. It starts the existing container, waits for Doris to become healthy, reconnects to the FE, and verifies the persisted row counts.

This restart cell is idempotent: it can be run when Docker Desktop and the container are stopped, starting, or already running. On macOS it opens Docker Desktop when necessary; on Linux, start Docker Engine before running the cell.


In [ ]:
lab.start_container("doris")

lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")
lab.sql("""
SELECT 'events' AS table_name, COUNT(*) AS recovered_rows FROM events
UNION ALL
SELECT 'events_v2', COUNT(*) FROM events_v2
UNION ALL
SELECT 'events_v3', COUNT(*) FROM events_v3
ORDER BY table_name
""", title="Recovered Lab 2 tables");

**Expected result:** the container returns to `healthy`, and all three tables report **10,158,080** rows.

## Lab complete

You kept one logical dataset constant while changing its physical design. Query Profile connected the leading sort key to BE scan work; `EXPLAIN` connected time partitions and Hash buckets to FE pruning; storage metadata connected load transactions to tablets, versions, Segments, and Compaction; and the final fixture demonstrated how each table model handles repeated keys.

The main distinctions are:

- A sort key and Prefix Index help the scan find relevant key ranges inside selected tablets.
- A partition boundary lets the FE exclude partitions.
- a Hash distribution key lets the FE exclude tablets for compatible equality predicates.
- Predicate pushdown evaluates filters in the scan operator; it does not by itself mean a partition or tablet was pruned.
- The Duplicate Key model, Unique Key model, and Aggregate Key model control repeated-key semantics.

Official references: [Duplicate Key model](https://doris.apache.org/docs/4.x/table-design/data-model/duplicate/) · [Auto Partitioning](https://doris.apache.org/docs/4.x/table-design/data-partitioning/auto-partitioning/) · [EXPLAIN](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/data-query/EXPLAIN/) · [Query Profile](https://doris.apache.org/docs/4.x/query-acceleration/query-profile/) · [Compaction](https://doris.apache.org/docs/4.x/key-features/data-compaction/)